prepare

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import re
import random

In [ ]:
# Import series of helper functions for the notebook

from helper_functions import create_tensorboard_callback, plot_loss_curves

In [ ]:
# Check GPU

tf.config.list_physical_devices('GPU')

Read dataset

In [ ]:
df_data=pd.read_csv('data_1A.csv')
df_data = df_data.drop(columns=['Unnamed: 0'])
df_data.head(10)

encode

In [ ]:
#Encoding Categorical to Numeric 

label_encode={"label": {"cardiovascular diseases":0,"general pathological conditions" :1,"nervous system diseases":2,
                       "digestive system diseases":3, "neoplasms":4}}
df_data=df_data.replace(label_encode)

cleansing text

In [ ]:
def cleansing(df_data):
    df_clean = df_data.str.lower()
    df_clean = [re.sub(r'[^\w\s]', '', i) for i in df_clean]
    df_clean = [re.sub(r'\s+', ' ', i) for i in df_clean]
    return df_clean

In [ ]:
df_data['clean_text'] = cleansing(df_data['text'])
df_data.drop('text', axis=1, inplace=True)
df_data.head(5)

Exploratory Data Analysis

In [ ]:
# Check unique

unique_sentiments = df_data['label'].unique()
print(unique_sentiments)

In [ ]:
# Check the distribution of classes

plt.figure(figsize=(10, 6))
sns.countplot(x='label', data=df_data)
plt.xticks(rotation=45)
plt.title('most class of text count')
plt.show()

In [ ]:
# Check the distribution of text lengths

df_data['text_length'] = df_data['clean_text'].apply(len)
plt.figure(figsize=(10, 6))
sns.histplot(df_data['text_length'])
plt.xlabel('Text Length')
plt.ylabel('Frequency')
plt.title('Distribution of Text Lengths')
plt.show()

In [ ]:
# visualize some random training examples
# create random indexes not higher than the total number of samples

random_index = random.randint(0, len(df_data) - 3)
for row in df_data[["clean_text", "label"]].iloc[random_index:random_index+3].itertuples():
    _, clean_text, label = row
    sentiment_label = ""
    if label == 0:
        sentiment_label = "(cardiovascular diseases)"
    elif label == 1:
        sentiment_label = "(general pathological conditions)"
    elif label == 2:
        sentiment_label = "(nervous system diseases)"
    elif label == 3:
        sentiment_label = "(digestive system diseases)"
    elif label == 4:
        sentiment_label = "(neoplasms)"
    
    print(f"Target: {label} {sentiment_label}")
    print(f"text:\n{clean_text}\n")
    print("---\n")

split test

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(df_data['clean_text'].to_numpy(),df_data['label'].to_numpy(),
    test_size=0.2,
    random_state=42,
    stratify=df_data['label']
)

In [ ]:
# Check the lengths

len(x_train), len(x_test), len(y_train), len(y_test)

In [ ]:
# View the sample training sentences and their labels

x_train[:3], y_train[:3]

Text vectorization (tokenization)

In [ ]:
# Find average number of tokens words in training
round(sum([len(i.split()) for i in x_train])/len(x_train))

In [ ]:
from tensorflow.keras.layers import TextVectorization

# Setup text vectorization with custom variables
max_vocab_length = 68000 # max number of words in vocabulary
max_length = 50 # max length 

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode="int",
                                    output_sequence_length=max_length)

In [ ]:
# Fit the text vectorizer to the training text

x_train = np.array(x_train)
text_vectorizer.adapt(x_train)

In [ ]:
# Create sample sentence and tokenize it

sample_sentence = "There's a flood in my street!"
text_vectorizer([sample_sentence])

In [ ]:
# Choose a random sentence from the training dataset and tokenize it

random_sentence = random.choice(x_train)
print(f"Original text:\n{random_sentence}\
      \n\nVectorized version:")
text_vectorizer([random_sentence])

Creating an Embedding using an Embedding Layer

In [ ]:
from tensorflow.keras import layers

embedding = layers.Embedding(input_dim=max_vocab_length, # set input shape
                             output_dim=128, # set size of embedding vector
                             embeddings_initializer="uniform", # default, intialize randomly
                             input_length=max_length, # how long is each input
                             name="embedding") 

embedding

In [ ]:
# Get a random sentence from training set

random_sentence = random.choice(x_train)
print(f"Original text:\n{random_sentence}\
      \n\nEmbedded version:")

# Embed the random sentence (turn it into numerical representation)
sample_embed = embedding(text_vectorizer([random_sentence]))
sample_embed

In [ ]:
# Check out a single token's embedding

sample_embed[0][0]

LSTM Model

In [ ]:
# Set random seed and embedding layer
tf.random.set_seed(42)

from tensorflow.keras import layers

inputs = layers.Input(shape=(1,), dtype="string")
x = text_vectorizer(inputs)
x = embedding(x)

# x = layers.LSTM
x = layers.LSTM(64)(x)

# x = layers.Dense
outputs = layers.Dense(5, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="model_LSTM")

In [ ]:
# Compile model
model.compile(loss="sparse_categorical_crossentropy",
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

In [ ]:
# Get a summary of the model
model.summary()

In [ ]:
# Create ModelCheckpoint callback
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_path = "./best_model.LSTM"
model_checkpoint = ModelCheckpoint(checkpoint_path, 
                                   monitor='val_accuracy', 
                                   save_best_only=True, 
                                   mode='max', 
                                   verbose=1)

In [ ]:
# Fit model
SAVE_DIR = "./logs"
model_history = model.fit(x_train,
                          y_train,
                          epochs=5,
                          validation_data=(x_test, y_test),
                          callbacks=[create_tensorboard_callback(SAVE_DIR, "LSTM_model"), model_checkpoint])

In [ ]:
plot_loss_curves(model_history)
plt.show()

In [ ]:
# Make predictions on the validation data

model_pred_probs = model.predict(x_test)
model_pred_probs.shape, model_pred_probs[:10]

In [ ]:
# Convert prediction probabilities to prediction classes

model_preds = tf.squeeze(tf.round(model_pred_probs))
model_preds[:10]

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def calculate_results(y_true, y_pred):

    # Convert one-hot encoded labels to class labels
    y_true_classes = np.argmax(y_true, axis=1) if len(y_true.shape) > 1 else y_true
    y_pred_classes = np.argmax(y_pred, axis=1) if len(y_pred.shape) > 1 else y_pred

    # Calculate model accuracy
    model_accuracy = accuracy_score(y_true_classes, y_pred_classes) * 100
    
    # Calculate model precision, recall, and f1 score using "weighted average"
    model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true_classes, y_pred_classes, average="weighted")
    model_results = {"accuracy": model_accuracy,
                     "precision": model_precision,
                     "recall": model_recall,
                     "f1": model_f1}
    return model_results

In [ ]:
model_results = calculate_results(y_true=y_test, y_pred=model_preds)
model_results

Machine learning - Naive Bayes

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Create a pipeline
model_ml = Pipeline([
  ("tf-idf", TfidfVectorizer()),
  ("clf", MultinomialNB())
])

# Fit the pipeline to the training data
model_ml.fit(X=x_train, 
            y=y_train);

In [ ]:
# Evaluate baseline on validation dataset

model_ml.score(X=x_test,
              y=y_test)

In [ ]:
# Make predictions
ml_preds = model_ml.predict(x_test)
ml_preds

In [ ]:
# Calculate baseline results
ml_preds = calculate_results(y_true=y_test,
                                     y_pred=ml_preds)
ml_preds

In [ ]:
from tensorflow.keras.models import load_model

# Load the best model
best_model = load_model("./best_model.LSTM")

# Evaluate the best model on the test data
evaluation_results = best_model.evaluate(x_test, y_test)

# Print the evaluation results
print("Test Accuracy:", evaluation_results[1])
print("Test Loss:", evaluation_results[0])